# Missing Data and Fallback Values

Real datasets often contain missing values. In this notebook, we will identify missing data, choose fallback values, replace missing values, and remove rows that cannot be used reliably.

---
## Learning objectives

By the end of this notebook, you will be able to:

- identify missing values with `isNull()` and `isNotNull()`;
- choose the first available value with `coalesce()`;
- replace missing values with `fillna()`; and
- remove unusable rows with `dropna()`.

---
## Set up customer-contact data

This DataFrame contains intentional missing values. In Spark, a missing value is shown as `NULL`. An empty string, such as `""`, is still text and is not a null value.

In [ ]:
from pyspark.sql import functions as F

customer_rows = [
    ("C001", "Northwind Supplies", "orders@northwind.example", None, "North"),
    ("C002", "Contoso Retail", None, "contact@contoso.example", "West"),
    ("C003", "Adventure Works", None, None, None),
    ("C004", "Fabrikam Stores", "sales@fabrikam.example", "hello@fabrikam.example", "South"),
    (None, "Unidentified Customer", None, "unknown@example", None),
]

customer_schema = """
    customer_id STRING,
    customer_name STRING,
    work_email STRING,
    personal_email STRING,
    region STRING
"""

customers = spark.createDataFrame(customer_rows, schema=customer_schema)
customers.show()

---
## Identify missing values

Use `isNull()` to create a condition for missing values and `isNotNull()` for values that are present. Both can be used inside `filter()`.

In [ ]:
customers_without_work_email = customers.filter(
    F.col("work_email").isNull()
)

customers_without_work_email.show()

customers_with_work_email = customers.filter(
    F.col("work_email").isNotNull()
)

customers_with_work_email.show()

---
## Choose a fallback value with `coalesce()`

`coalesce()` returns the first value that is not null. Here, a work email is preferred; if it is missing, Spark uses the personal email instead. If both are missing, `preferred_email` is null.

In [ ]:
customers_with_preferred_email = customers.withColumn(
    "preferred_email",
    F.coalesce(
        F.col("work_email"),
        F.col("personal_email"),
    ),
)

customers_with_preferred_email.show()

---
## Replace missing values with `fillna()`

Use `fillna()` when one known replacement makes sense for a column. It returns a new DataFrame and leaves the original DataFrame unchanged.

In [ ]:
customers_with_region = customers.fillna({"region": "Unknown"})
customers_with_region.show()

---
## Remove unusable rows with `dropna()`

Some fields are essential. A missing `customer_id` means the row cannot be joined reliably to customer or order data, so we remove rows where that key is missing.

In [ ]:
customers_with_valid_id = customers.dropna(subset=["customer_id"])
customers_with_valid_id.show()

---
## Your turn

**Exercise 1:** Create a DataFrame named `customers_without_work_email` containing only customers who have no work email. Preview it with `show()`.

In [ ]:
# Write your solution here.

**Exercise 2:** Create a DataFrame named `customers_clean`. It must remove rows with a missing `customer_id`, replace missing `region` values with `Unknown`, and add `preferred_email` using the first available work or personal email. Preview it with `show()`.

In [ ]:
# Write your solution here.

---
## Next lesson

Next, we will group rows and calculate summary measures such as total revenue and order count.